# Stage 3: Neural Reranking

This stage takes the 5000-passage shortlist produced by Stage 2 and narrows it down to a small ranked list of the most relevant evidences for each claim, using two neural rerankers in cascade:

```
1.2M  →  BM25 + keywords  →  5 000  →  Bi-encoder  →  200  →  Cross-encoder  →  top-30 (with scores)
          (Stage 2)                     (Stage 3a)         (Stage 3b)
```

- **Bi-encoder** — `all-MiniLM-L6-v2`, fine-tuned with in-batch contrastive loss plus one hard negative per claim sampled from BM25 ranks 50–500. Cheap to score, so we use it to filter 5000 → 200.
- **Cross-encoder** — `cross-encoder/ms-marco-MiniLM-L-6-v2`, fine-tuned with a pairwise margin loss (`max(0, m − s(pos) + s(neg))`) over hard negatives sampled from bi-encoder ranks 30–200. This is the more accurate but slower reranker; it scores the 200 survivors.

We save the cross-encoder's **scores** alongside the IDs so that Stage 5 can pick its own evidence cut-off (fixed top-k or adaptive thresholding) based on the score distribution.

## 1. Setup

Import PyTorch, the relevant transformer classes, and set a fixed random seed. Configure model names, sequence lengths, and rerank-size hyperparameters.

In [ ]:
import sys

sys.path.insert(0, "../src")  # repo-local package
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
from factcheck import config as cfg
from factcheck.config import load_json as load
from factcheck.metrics import f1_at, recall_at

In [ ]:
PROC_DIR, SL_DIR, RERANK_DIR, MODEL_DIR = cfg.PROC_DIR, cfg.SL_DIR, cfg.RERANK_DIR, cfg.MODEL_DIR
for d in (RERANK_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Models — both small enough for free Colab T4
BI_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Stage 3a: bi-encoder filtering 5000 -> 200
TOP_AFTER_BI = 200
MAX_LEN_BI = 128

# Stage 3b: cross-encoder rerank 200 -> stored full ranking (top-k chosen at stage 5)
TOP_AFTER_CROSS = 30  # save top-30 cross-encoder scores per claim for stage 5
MAX_LEN_CROSS = 256

## 2. Load data

Read the BERT-cleaned evidence corpus from Stage 1 and the three shortlists from Stage 2.

In [ ]:
evidence_bert = load(PROC_DIR / "evidence_bert.json")
train_claims = load(PROC_DIR / "train_claims_proc.json")
dev_claims = load(PROC_DIR / "dev_claims_proc.json")
test_claims = load(PROC_DIR / "test_claims_proc.json")

train_sl = load(SL_DIR / "train_shortlist.json")
dev_sl = load(SL_DIR / "dev_shortlist.json")
test_sl = load(SL_DIR / "test_shortlist.json")
print("sample shortlist size:", len(next(iter(dev_sl.values()))))

## 3. Bi-encoder fine-tuning

Fine-tune the bi-encoder with in-batch contrastive learning: each batch contains a claim, its positive evidence, and one hard negative sampled from BM25 ranks 50–500 (low enough to still be topically related, high enough to almost certainly be wrong). Within a batch of size B, every claim has 2B candidate evidences (B positives + B hard negatives) and the cross-entropy loss pushes its score for the true positive above all the others. Hard negatives are re-sampled at every epoch to widen exposure.

`MeanPoolBiEncoder` does mean-pooling over the last hidden state (masked by `attention_mask`) and L2-normalises, so the similarity is a cosine.

In [ ]:
class BiPairsDataset(Dataset):
    """(claim, gold evidence, hard negative) triples.

    Hard negatives are resampled every epoch from BM25 shortlist positions
    [HN_LOW:HN_HIGH] excluding gold: topically related but wrong."""

    HN_LOW, HN_HIGH = 50, 500

    def __init__(self, claims, evidence_dict, shortlists):
        self.evidence = evidence_dict
        self.examples = []
        for cid, c in claims.items():
            gold = set(c["evidences"])
            hard_pool = [
                e for e in shortlists[cid][self.HN_LOW : self.HN_HIGH] if e in evidence_dict and e not in gold
            ]
            if not hard_pool:
                continue
            for ev in c["evidences"]:
                if ev in evidence_dict:
                    self.examples.append((c["claim_bert"], evidence_dict[ev], hard_pool))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        cl, pos_text, pool = self.examples[i]
        return cl, pos_text, self.evidence[pool[random.randrange(len(pool))]]


bi_dataset = BiPairsDataset(train_claims, evidence_bert, train_sl)
print(f"bi-encoder training triples: {len(bi_dataset):,}")

In [ ]:
class MeanPoolBiEncoder(nn.Module):
    def __init__(self, name=BI_MODEL_NAME):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(name)

    def encode(self, ids, mask):
        out = self.backbone(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).float()
        pooled = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
        return F.normalize(pooled, dim=-1)


bi_tok = AutoTokenizer.from_pretrained(BI_MODEL_NAME)


def bi_collate(batch):
    cs, ps, ns = zip(*batch)
    cE = bi_tok(list(cs), padding=True, truncation=True, max_length=MAX_LEN_BI, return_tensors="pt")
    pE = bi_tok(list(ps), padding=True, truncation=True, max_length=MAX_LEN_BI, return_tensors="pt")
    nE = bi_tok(list(ns), padding=True, truncation=True, max_length=MAX_LEN_BI, return_tensors="pt")
    return cE, pE, nE

In [ ]:
def train_bi(epochs=5, batch_size=32, lr=2e-5, temperature=0.05):
    model = MeanPoolBiEncoder().to(DEVICE)
    loader = DataLoader(bi_dataset, batch_size=batch_size, shuffle=True, collate_fn=bi_collate, num_workers=0)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))
    ce = nn.CrossEntropyLoss()

    model.train()
    for ep in range(epochs):
        running = 0.0
        for cE, pE, nE in tqdm(loader, desc=f"bi ep{ep + 1}"):
            cE = {k: v.to(DEVICE) for k, v in cE.items()}
            pE = {k: v.to(DEVICE) for k, v in pE.items()}
            nE = {k: v.to(DEVICE) for k, v in nE.items()}
            opt.zero_grad()
            with autocast("cuda", enabled=(DEVICE == "cuda")):
                c_emb = model.encode(cE["input_ids"], cE["attention_mask"])
                p_emb = model.encode(pE["input_ids"], pE["attention_mask"])
                n_emb = model.encode(nE["input_ids"], nE["attention_mask"])
                # all evidence embeddings as the candidate pool: positives + in-batch positives + this batch's hard negs
                ev_emb = torch.cat([p_emb, n_emb], dim=0)  # (2B, h)
                logits = (c_emb @ ev_emb.t()) / temperature  # (B, 2B)
                labels = torch.arange(c_emb.size(0), device=DEVICE)  # positives are at [0..B-1]
                loss = ce(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
        print(f"  ep{ep + 1} avg loss = {running / len(loader):.4f}")
    torch.save(model.state_dict(), MODEL_DIR / "bi_encoder.pt")
    return model


bi_model = train_bi(epochs=5)

## 3b. Apply the bi-encoder: rerank 5000 → 200

For every claim, encode the claim once and all 5000 candidates in batches, then take the top-200 by dot product. The recall numbers printed below show how much of the gold evidence survives this narrowing.

In [ ]:
@torch.no_grad()
def bi_embed_batch(texts, model, batch_size=256):
    model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i : i + batch_size]
        enc = bi_tok(chunk, padding=True, truncation=True, max_length=MAX_LEN_BI, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        emb = model.encode(enc["input_ids"], enc["attention_mask"]).cpu().numpy()
        out.append(emb)
    return np.concatenate(out, 0)


def bi_rerank(claims, shortlists, model, top_k=TOP_AFTER_BI):
    out = {}
    for cid, c in tqdm(claims.items(), desc="bi-rerank"):
        cand_ids = shortlists[cid]
        cand_texts = [evidence_bert[e] for e in cand_ids if e in evidence_bert]
        if not cand_texts:
            out[cid] = cand_ids[:top_k]
            continue
        c_emb = bi_embed_batch([c["claim_bert"]], model)
        e_emb = bi_embed_batch(cand_texts, model)
        scores = (c_emb @ e_emb.T).squeeze(0)
        order = np.argsort(-scores)[:top_k]
        out[cid] = [cand_ids[i] for i in order]
    return out


train_bi = bi_rerank(train_claims, train_sl, bi_model)
dev_bi = bi_rerank(dev_claims, dev_sl, bi_model)
test_bi = bi_rerank(test_claims, test_sl, bi_model)

In [ ]:
for k in (10, 30, 50, 100, 200):
    print(f"dev recall@{k:<4d} after bi-encoder = {recall_at(dev_bi, dev_claims, k):.4f}")

## 4. Cross-encoder — pairwise margin training

The cross-encoder scores `(claim, evidence)` pairs jointly (full attention across both texts). It's initialised from the MS-MARCO-tuned checkpoint, which already has a 1-dimensional regression head, so we just fine-tune end-to-end.

Training uses a **pairwise margin loss** rather than binary cross-entropy. For each claim we sample a hard negative from bi-encoder ranks 30–200 — passages the bi-encoder already ranked plausibly highly, which makes them informative training signal. The loss `max(0, m − s(pos) + s(neg))` only requires the positive to outscore the negative by at least `margin`, leaving the model free to rank the rest of the shortlist however it likes.

In [ ]:
class CrossPairwiseDataset(Dataset):
    """Pairwise (claim, gold_evidence, hard_neg_evidence).
    Hard negatives are sampled from positions 30..200 of the **bi-encoder rerank** —
    bi-encoder thinks they're plausible, so they're informative for the cross-encoder."""

    HN_LOW, HN_HIGH = 30, 200

    def __init__(self, claims, evidence_dict, bi_rerank):
        self.examples = []
        for cid, c in claims.items():
            golds = [e for e in c["evidences"] if e in evidence_dict]
            gold_set = set(c["evidences"])
            hard_pool = [
                e
                for e in bi_rerank[cid][self.HN_LOW : self.HN_HIGH]
                if e in evidence_dict and e not in gold_set
            ]
            if not hard_pool:
                continue
            for ev in golds:
                self.examples.append((c["claim_bert"], evidence_dict[ev], hard_pool))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        cl, pos_text, pool = self.examples[i]
        neg_id = pool[random.randrange(len(pool))]
        return cl, pos_text, evidence_bert[neg_id]


cross_ds = CrossPairwiseDataset(train_claims, evidence_bert, train_bi)
print(f"cross pairs: {len(cross_ds):,}")

In [ ]:
cross_tok = AutoTokenizer.from_pretrained(CROSS_MODEL_NAME)


def cross_collate(batch):
    cs, ps, ns = zip(*batch)
    pos = cross_tok(
        list(cs), list(ps), padding=True, truncation=True, max_length=MAX_LEN_CROSS, return_tensors="pt"
    )
    neg = cross_tok(
        list(cs), list(ns), padding=True, truncation=True, max_length=MAX_LEN_CROSS, return_tensors="pt"
    )
    return pos, neg


class CrossScorer(nn.Module):
    """Cross-encoder with a 1-dim regression head."""

    def __init__(self, name=CROSS_MODEL_NAME):
        super().__init__()
        # use SequenceClassification with num_labels=1 — that's what the MS-MARCO ckpt was trained for,
        # so we KEEP the original head's weights and just fine-tune.
        self.backbone = AutoModelForSequenceClassification.from_pretrained(name, num_labels=1)

    def forward(self, **kw):
        return self.backbone(**kw).logits.squeeze(-1)  # (B,)

In [ ]:
def train_cross(epochs=3, batch_size=16, lr=2e-5, margin=1.0):
    model = CrossScorer().to(DEVICE)
    loader = DataLoader(
        cross_ds, batch_size=batch_size, shuffle=True, collate_fn=cross_collate, num_workers=0
    )
    opt = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))
    model.train()
    for ep in range(epochs):
        running = 0.0
        for pos, neg in tqdm(loader, desc=f"cross ep{ep + 1}"):
            pos = {k: v.to(DEVICE) for k, v in pos.items()}
            neg = {k: v.to(DEVICE) for k, v in neg.items()}
            opt.zero_grad()
            with autocast("cuda", enabled=(DEVICE == "cuda")):
                s_pos = model(**pos)
                s_neg = model(**neg)
                # Pairwise margin: gold should beat hard negative by `margin`
                loss = F.relu(margin - s_pos + s_neg).mean()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
        print(f"  ep{ep + 1} avg loss = {running / len(loader):.4f}")
    torch.save(model.state_dict(), MODEL_DIR / "cross_encoder.pt")
    return model


cross_model = train_cross(epochs=3, batch_size=16, lr=2e-5)

## 4b. Rerank bi-encoder top-200 and save scores

Run the cross-encoder over the 200-passage bi-encoder output for each claim, store the top-30 IDs together with their **raw scores**, and write the result to `<split>_cross_topk.json`. Keeping the scores (not just the IDs) lets Stage 5 try both fixed top-k and adaptive thresholding strategies without rerunning the cross-encoder.

In [ ]:
@torch.no_grad()
def cross_score(claim_text, ev_texts, model, bs=64):
    model.eval()
    all_s = []
    for i in range(0, len(ev_texts), bs):
        chunk = ev_texts[i : i + bs]
        enc = cross_tok(
            [claim_text] * len(chunk),
            chunk,
            padding=True,
            truncation=True,
            max_length=MAX_LEN_CROSS,
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            s = model(**enc).float().cpu().numpy()
        all_s.append(s)
    return np.concatenate(all_s, 0)


def cross_rerank_with_scores(claims, bi_ranking, model, top_k=TOP_AFTER_CROSS):
    out = {}
    for cid, c in tqdm(claims.items(), desc="cross"):
        cand_ids = bi_ranking[cid]
        cand_texts = [evidence_bert[e] for e in cand_ids if e in evidence_bert]
        if not cand_texts:
            out[cid] = {"ids": cand_ids[:top_k], "scores": [0.0] * min(top_k, len(cand_ids))}
            continue
        scores = cross_score(c["claim_bert"], cand_texts, model)
        order = np.argsort(-scores)[:top_k]
        out[cid] = {
            "ids": [cand_ids[i] for i in order],
            "scores": [float(scores[i]) for i in order],
        }
    return out


train_final = cross_rerank_with_scores(train_claims, train_bi, cross_model)
dev_final = cross_rerank_with_scores(dev_claims, dev_bi, cross_model)
test_final = cross_rerank_with_scores(test_claims, test_bi, cross_model)

## 5. Ablation: BM25 vs +Bi vs +Cross

Compute recall@k and F1@k on dev at each stage of the pipeline. This is the table that quantifies each reranker's contribution to the final retrieval quality.

In [ ]:
def truncate_scored(d, k):
    return {cid: v["ids"][:k] for cid, v in d.items()}


print(f"{'Stage':<28s} | {'r@4':>6s} {'r@5':>6s} {'r@10':>6s} | {'F1@4':>6s} {'F1@5':>6s}")
print("-" * 75)
for name, d in [
    ("BM25+NER only", dev_sl),
    ("+ Bi-encoder", dev_bi),
    ("+ Cross-encoder", truncate_scored(dev_final, 200)),
]:  # full ranking for r@
    r4 = recall_at(d, dev_claims, 4)
    r5 = recall_at(d, dev_claims, 5)
    r10 = recall_at(d, dev_claims, 10)
    f4 = f1_at(d, dev_claims, 4)
    f5 = f1_at(d, dev_claims, 5)
    print(f"{name:<28s} | {r4:.4f} {r5:.4f} {r10:.4f} | {f4:.4f} {f5:.4f}")

## 6. Evidence cut-off strategies

The evidence F-score depends on how many evidences we return per claim. Gold averages 3.36 evidences/claim, so the F-optimal fixed k is somewhere in {3, 4, 5}. We try both:

- **Fixed k**: return the same number of evidences for every claim. Simple but ignores classifier confidence.
- **Adaptive**: return everything whose cross-encoder score is at or above a threshold τ (clipped between 1 and 5 items per claim). τ is chosen on dev by sweeping its quantile over all scores.

Stage 5 makes the final pick using both retrieval F1 and downstream classification accuracy, so for this notebook we only report the F-score behaviour.

In [ ]:
print(f"{'k':<6s} | {'F1':>6s} {'recall':>6s}")
for k in (1, 2, 3, 4, 5, 6, 7, 8):
    d_k = truncate_scored(dev_final, k)
    print(f"{k:<6d} | {f1_at(d_k, dev_claims, k):.4f} {recall_at(d_k, dev_claims, k):.4f}")

In [ ]:
# Adaptive: keep score >= threshold; clamp into [1, max_k]
def adaptive_topk(scored, threshold, min_k=1, max_k=5):
    out = {}
    for cid, v in scored.items():
        ids, sc = v["ids"], v["scores"]
        if not sc:
            out[cid] = ids[:min_k]
            continue
        # Take items where score >= threshold, between min_k..max_k items
        chosen = [eid for eid, s in zip(ids, sc) if s >= threshold]
        if len(chosen) < min_k:
            chosen = ids[:min_k]
        if len(chosen) > max_k:
            chosen = chosen[:max_k]
        out[cid] = chosen
    return out


# Sweep threshold quantiles
all_scores = [s for v in dev_final.values() for s in v["scores"]]
for q in (0.50, 0.70, 0.80, 0.85, 0.90, 0.92, 0.95):
    tau = float(np.quantile(all_scores, q))
    sl = adaptive_topk(dev_final, tau, min_k=1, max_k=5)
    avg_k = np.mean([len(v) for v in sl.values()])
    print(f"quantile={q:.2f}  tau={tau:+.3f}  avg_k={avg_k:.2f}  F1={f1_at(sl, dev_claims):.4f}")

## 7. Save outputs for Stages 4 and 5

Two files are written per split: the rich `<split>_cross_topk.json` (top-30 IDs + scores, used by Stage 5 for adaptive thresholding) and a thinner `<split>_top4.json` (just the top-4 IDs, used by Stage 4's classifier as the default evidence input).

In [ ]:
for name, d in [("train", train_final), ("dev", dev_final), ("test", test_final)]:
    p = RERANK_DIR / f"{name}_cross_topk.json"
    with open(p, "w") as f:
        json.dump(d, f)
    print(f"wrote {p}")

# Backward compatibility: also write a plain top-4 id list (used by stage 4)
for name, d in [("train", train_final), ("dev", dev_final), ("test", test_final)]:
    p = RERANK_DIR / f"{name}_top4.json"
    with open(p, "w") as f:
        json.dump({cid: v["ids"][:4] for cid, v in d.items()}, f)

## Summary

The notebook trains and applies two neural rerankers in cascade, and writes four sets of outputs per split: shortlists, bi-encoder ranks, cross-encoder ranks with scores, and a convenience top-4 list. The bi-encoder strips the shortlist down to a tractable size for the cross-encoder, and the cross-encoder produces the final ranking that Stage 4's classifier and Stage 5's pipeline consume.